# Lab 0 · Next-token con una GRU

Material del **motor** (ebook cap. 2). No es un tutorial de PyTorch: es el examen
para que Qwen no sea una caja negra.

**Pregunta del lab.** ¿Quién pone la etiqueta? Nadie. La etiqueta es *la letra que sí venía después* en el Quijote.

## Dummy (antes de la fórmula)

Imagina que lees en voz alta y, en cada letra, apuestas qué sigue. Si el texto dice `Banxi`, tu apuesta debería ser `c` (de *Banxico*). Ganas puntos si le diste alta probabilidad a esa letra. Pierdes si le diste 1% y se la diste a `t` (*tata*). Un LLM es esa máquina de apostar, repetida millones de veces.

## Matemática (la misma idea, en serio)

Un modelo de lenguaje factoriza la probabilidad de un texto con la **regla de la cadena**:

$$
P_\theta(x_1,\ldots,x_T)=\prod_{t=1}^{T} P_\theta(x_t\mid x_{<t}).
$$

Entrenar es **maximizar** esa probabilidad (máxima verosimilitud) o, equivalente y más cómodo para PyTorch, **minimizar** la entropía cruzada:

$$
\mathcal{L}(\theta)=-\frac{1}{BT}\sum_{b=1}^{B}\sum_{t=1}^{T}\log P_\theta\bigl(y_{b,t}\mid x_{b,\le t}\bigr).
$$

$B$ = cuántas ventanas van en el batch. $T$ = letras por ventana. El `view(B*T, V)` de más abajo *es* esa doble suma aplanada: no es magia.

En cristiano: recorres el Quijote letra por letra. En cada posición preguntas “¿qué sigue?” y bajas puntos si no acierta la letra correcta. $0$ de pérdida sería adivinar siempre con certeza $1$ (imposible). Un modelo que no aprendió nada anda cerca de $\log |V|$ (el tamaño del abecedario).

> Ebook: [El motor, a mano](https://eda2-ebook-509314519015.us-central1.run.app/llms/) · repo canónico de este notebook.


## 0 · Cómo correr esto sin sufrimiento

| Dónde | Qué hacer |
|---|---|
| **Local** | `git clone` este repo. El archivo `quijote.txt` ya está. `pip install torch`. |
| **Colab** | Abre el `.ipynb` desde GitHub. Descomenta `%pip` y `wget` en la celda de setup. |

En **CPU** dejamos el modelo chico (`hidden_dim=128`, 1 capa, 3 épocas) y **no** recorremos las ~2 millones de ventanas: muestreamos 8 000. Es el mismo objetivo, menos filas. En GPU puedes subir.

Si ves `FileNotFoundError: quijote.txt`, o no clonaste el repo, o en Colab no corriste el `wget`.


In [ ]:
# Colab: descomenta las dos líneas de abajo (una vez).
# %pip install -q torch
# !wget -q https://raw.githubusercontent.com/NoSoyRo/diplomado-ia-llms/main/quijote.txt

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import List, Optional, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TEXT_PATH = "quijote.txt"
print(f"device={DEVICE}  archivo={TEXT_PATH}")


## 1 · Vocabulario: letras ↔ números

**Dummy.** La red no come letras: come enteros. `stoi['a'] = 7` es el diccionario de ida; `itos[7] = 'a'` es el de vuelta. Si se te pierde uno, el modelo habla basura.

**Math.** Sea $V$ el conjunto de caracteres observados. Los mapas

$$
\mathrm{stoi}:V\to\{0,\ldots,|V|-1\},\qquad \mathrm{itos}=\mathrm{stoi}^{-1}
$$

son biyecciones. En este lab un token = **un carácter**. Transparencia > realismo. El cuchillo de verdad (BPE) llega después, con Qwen.

El vocabulario crece si el texto trae acentos, `ñ`, comillas raras. No los borres: el Quijote *es* eso.


In [ ]:
def load_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read().replace("\r\n", "\n").replace("\r", "\n")


def build_vocab(text: str):
    chars = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    return stoi, itos


def encode(text: str, stoi: dict) -> torch.Tensor:
    return torch.tensor([stoi[ch] for ch in text], dtype=torch.long)


def decode(ids: List[int], itos: dict) -> str:
    return "".join(itos[i] for i in ids)


## 2 · Ventanas $x$ / $y$: aquí está la etiqueta

**Dummy.** Cortas un trozo de $L$ letras y el “examen” es el mismo trozo corrido una letra.

Ejemplo: `x = Banxi`, `y = anxic`. Nadie etiquetó eso a mano. Se llama **auto-supervisión**: la respuesta correcta es “lo que sí venía después”.

**Math.** Con longitud $L$, el ejemplo $i$ es

$$
x^{(i)}=(x_i,\ldots,x_{i+L-1}),\qquad y^{(i)}=(x_{i+1},\ldots,x_{i+L}),
$$

es decir $y^{(i)}_k = x^{(i)}_{k+1}$.

Señala esta clase el día del lab. Si dices “un humano puso la etiqueta”, se repite la sesión.

`max_windows` no cambia la fórmula: solo elige un subconjunto de inicios $i$ al azar para que CPU no se muera.


In [ ]:
class CharWindowDataset(Dataset):
    """Ventanas x → y corrido +1. Si max_windows no es None, muestrea inicios al azar.

    Dummy: el Quijote tiene ~2M letras. Recorrer TODAS las ventanas en CPU no cabe
    en una sesión. Muestrear M inicios es el mismo examen, con menos filas.
    """

    def __init__(self, data_ids: torch.Tensor, seq_len: int, max_windows: Optional[int] = 8000, seed: int = 0):
        self.data = data_ids
        self.seq_len = seq_len
        n = max(0, len(self.data) - self.seq_len - 1)
        if max_windows is not None and n > max_windows:
            g = torch.Generator().manual_seed(seed)
            self.index = torch.randint(0, n, (max_windows,), generator=g)
        else:
            self.index = torch.arange(n)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i: int):
        idx = int(self.index[i])
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + 1 : idx + self.seq_len + 1]
        return x, y


## 3 · GRU: todo el pasado en un vaso

**Dummy.** Una GRU no “recuerda el capítulo”. Comprime todo lo leído en un vector $h_t$ de dimensión fija (128, 256…). Ese vector es un **vaso**. Cerca del borde funciona; lejos se olvida. Por eso “Banxico” al principio del párrafo se evapora cuando llegas al verbo.

Arquitectura de este lab: `Embedding → GRU → Linear` hacia $|V|$ logits.

**Math (Cho et al., 2014).** Con reset $r_t$, update $z_t$ y candidato $\tilde h_t$:

$$
\begin{aligned}
z_t&=\sigma(W_z x_t+U_z h_{t-1}),\\
r_t&=\sigma(W_r x_t+U_r h_{t-1}),\\
\tilde h_t&=\tanh(W x_t+U(r_t\odot h_{t-1})),\\
h_t&=(1-z_t)\odot h_{t-1}+z_t\odot\tilde h_t.
\end{aligned}
$$

No memorices las cuatro líneas para recitarlas. Entiéndelas como un vaso con dos llaves:

- $z_t$ (update) decide cuánto del pasado se queda y cuánto se reemplaza.
- $r_t$ (reset) decide cuánto del pasado se usa para inventar el candidato.

El logit es $W_o h_t$: una capa lineal del vaso al abecedario. Si $d=128$, *todo* —el titular de hace dos párrafos, el sujeto, la coma— tiene que caber en 128 números. El lab 2 existe para oír que ese cuello de botella no se arregla alargando $L$.


In [ ]:
class CharGRULM(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int = 128, hidden_dim: int = 256, num_layers: int = 2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.gru = nn.GRU(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, h0=None):
        # x: (B, T) → logits (B, T, V) y vaso final hN
        emb = self.embed(x)
        out, hN = self.gru(emb, h0)
        logits = self.fc(out)
        return logits, hN


## 4 · Entrenar: cross-entropy, clip, Adam

**Dummy.** Cada batch: predice, compara con $y$, empuja los pesos un poquito para que la próxima vez le pegue más a la letra correcta. Adam es el que decide el tamaño del empujón. El *grad clipping* a $1.0$ es un seguro: si un batch sale loco, no explota el entrenamiento.

Mira **dos** curvas, no una:

| Número | Qué es | Para qué |
|---|---|---|
| `train_loss` | qué tan mal predice en el texto con el que estudia | siempre *puede* bajar (memoriza) |
| `val_loss` | qué tan mal predice en un pedazo que **no** usó | el que importa |
| perplejidad | $e^{\mathrm{val\_loss}}$ | “de cuántos tokens dudaba, en promedio” |

**Math.** Si la val_loss es $2.3$ nats, $\mathrm{PPL}=e^{2.3}\approx 10$: como si en cada paso dudara entre diez letras. Un modelo que memorizó el Quijote puede bajar a $3$; uno virgen anda por $|V|$.

Overfitting: train sigue bajando y val **sube**. Ya no aprende español: se aprendió el libro. En un noticiero eso es mortal (el 15 de agosto el corpus de julio no sirve).

$$
\mathrm{PPL}(P_\theta)=\exp(\mathcal{L}_{\mathrm{val}}).
$$

Más baja = mejor ajuste. **No** = briefing más honesto. Un modelo puede bajar PPL y alucinar más, porque aprendió a sonar seguro.


In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    n_batches = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        logits, _ = model(x)
        B, T, V = logits.shape
        loss = criterion(logits.view(B * T, V), y.view(B * T))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(1, n_batches)


@torch.no_grad()
def eval_loss(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    n_batches = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits, _ = model(x)
        B, T, V = logits.shape
        loss = criterion(logits.view(B * T, V), y.view(B * T))
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(1, n_batches)


## 5 · Generar: temperatura y el loop que se come a sí mismo

**Dummy.** “Hablar” no es un botón mágico. Es un loop: predice una letra, la pega al texto, predice otra vez. Por eso es **auto-regresivo**: se condiciona a lo que él mismo escupió. Si en el paso 3 salió `tata` en vez de `tasa`, el paso 4 ya vive en ese universo.

La **temperatura** $T$ es un volumen. Divides los logits entre $T$ *antes* del softmax.

| $T$ | Qué pasa |
|---|---|
| baja ($0.3$–$0.7$) | texto seguro, aburrido, a veces `el el el` |
| $1.0$ | respeta las probabilidades del modelo |
| alta ($1.2+$) | más diverso, más incoherente |

No hay una $T$ correcta. Hay un oficio: el noticiero quiere baja; un poema, más alta.

**Math.** Si $z\in\mathbb{R}^{|V|}$ son los logits,

$$
P_T(v)=\frac{\exp(z_v/T)}{\sum_u \exp(z_u/T)}.
$$

$T\to 0$ concentra en $\arg\max z$ (greedy). $T\to\infty$ se aplana hacia la uniforme. El loop es

$$
x_{t+1}\sim P_\theta(\cdot\mid x_{\le t}).
$$

Un error temprano entra al contexto y sesga todo lo que sigue (exposición al sesgo). El briefing que se tuerce al principio **no** se corrige solo más adelante.


In [ ]:
@torch.no_grad()
def generate(model, prompt: str, stoi: dict, itos: dict, max_new_tokens: int = 400, temperature: float = 1.0, device="cpu"):
    """Auto-regresivo: predice, pega, predice otra vez.

    El primer token nuevo sale de los logits del ÚLTIMO carácter del prompt.
    (Antes el código volvía a comer ese carácter: el texto salía corrido.)
    """
    model.eval()
    ids = [stoi[ch] for ch in prompt if ch in stoi] or [0]
    out_ids = ids.copy()
    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    logits, h = model(x, h0=None)
    for _ in range(max_new_tokens):
        z = logits[0, -1, :] / max(1e-8, temperature)
        probs = torch.softmax(z, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1).item()
        out_ids.append(next_id)
        inp = torch.tensor([[next_id]], dtype=torch.long, device=device)
        logits, h = model(inp, h0=h)
    return decode(out_ids, itos)


## 6 · Corre el ciclo y no te quedes en el número

Hold-out $90/10$ por **prefijo** (las últimas letras son val). No es perfecto —el Quijote no es i.i.d.— pero evita el autoengaño de evaluar en el mismo párrafo.

Cuando termine:

1. Copia la tabla época / train / val / ppl.
2. Genera **el mismo prompt** a tres temperaturas.
3. Responde en una frase: ¿qué patrones aprendió (n-gramas, puntuación, “Quijote”, “Sancho”) y qué no?

Señala en el código dónde está $y$. Si no puedes, vuelve a la celda 2.


In [ ]:
@dataclass
class TrainConfig:
    seq_len: int = 64
    batch_size: int = 32
    emb_dim: int = 128
    hidden_dim: int = 128
    num_layers: int = 1
    lr: float = 3e-4
    epochs: int = 3
    max_windows: int = 8000
    device: str = DEVICE


def main(text_path: str, cfg: TrainConfig):
    text = load_text(text_path)
    stoi, itos = build_vocab(text)
    data_ids = encode(text, stoi)

    n = len(data_ids)
    split = int(0.9 * n)
    train_ids, val_ids = data_ids[:split], data_ids[split:]

    train_ds = CharWindowDataset(train_ids, cfg.seq_len, max_windows=cfg.max_windows, seed=0)
    val_ds = CharWindowDataset(val_ids, cfg.seq_len, max_windows=min(2000, cfg.max_windows), seed=1)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, drop_last=True)

    device = torch.device(cfg.device)
    model = CharGRULM(len(stoi), cfg.emb_dim, cfg.hidden_dim, cfg.num_layers).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    print(f"Vocab={len(stoi)} | train_windows={len(train_ds)} | val_windows={len(val_ds)}")
    for epoch in range(1, cfg.epochs + 1):
        tr = train_one_epoch(model, train_loader, optimizer, device)
        vl = eval_loss(model, val_loader, device)
        ppl = math.exp(vl) if vl < 20 else float("inf")
        print(f"Epoch {epoch:02d} | train_loss={tr:.4f} | val_loss={vl:.4f} | val_ppl={ppl:.2f}")

    prompt = "En un lugar de la Mancha "
    for temp in [0.7, 1.0, 1.2]:
        sample = generate(model, prompt, stoi, itos, max_new_tokens=300, temperature=temp, device=device)
        print("\n" + "=" * 60)
        print(f"Temperature = {temp}")
        print(sample)

    return model, stoi, itos


cfg = TrainConfig()
print("GPU: sube hidden_dim=256, num_layers=2, seq_len=128, epochs=5.")
model, stoi, itos = main(TEXT_PATH, cfg)
